# Giving Reachy objects to work with

`FWDCenterLabSivaPool.yaml` holds a **pool** of ten objects parked on the floor
beside the table, and an **empty board**.  This notebook puts them on named grid
cells while the simulation runs, so a run can decide what Reachy is looking at
instead of inheriting a fixed layout.

**Why a pool, rather than creating objects on demand.**  MuJoCo freezes
`nbody`/`njnt`/`ngeom` at compile time — there is no `add_body()`, which is why
`server.py` still refuses a runtime `scene_load`.  Every object that will ever
exist has to be declared in the scene.  Declaring spares up front and moving
them is what "add an object at runtime" actually means here.

Pick-and-place work uses `FWDCenterLabSiva.yaml`, which keeps its four
manipulables on the grid.  This scene is the one where the board starts empty.

Start the sim on this scene first:

```
REACHY_SIM_SCENE=FWDCenterLabSivaPool REACHY_SIM_DISTORTION=1 ./scripts/start_sim.sh
```

Watch it at **RViz** http://localhost:6080 or **camera** http://localhost:8080.

In [1]:
import asyncio, json, sys, pathlib
import websockets

REPO = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO / "native_mujoco"))
sys.path.insert(0, str(REPO / "src"))

from protocol import Hello, PlaceObject

URI = "ws://127.0.0.1:8765"
SCENE = REPO / "scenes" / "FWDCenterLabSivaPool.yaml"

## 1. The board

Read from the same YAML the simulator loaded, so the notebook and the physics
cannot disagree about where a cell is.

`SceneModel.from_yaml` follows `extends:`, so this sees the parent's board and
grid — not just what this child restates.  It did not always: until 2026-09-09
it used a raw YAML load, and a child scene arrived with no table, no cells and
no rails.

In [2]:
from reachy_ai.scene.awareness import SceneModel
from placement import cells_from_scene, pool_ids
from scene_io import load_scene

scene = SceneModel.from_yaml(str(SCENE))
doc = load_scene(str(SCENE))
cells = cells_from_scene(doc)

print(f"table surface z : {scene.table_surface_z:.4f} m")
print(f"rig rails       : {len([o for o in scene.static_obstacles() if 'rig-frame' in o.tags])}")
print()
print("cells (row 1 = nearest the robot, col 1 = its LEFT / +y):")
for name in sorted(cells):
    c = cells[name]
    flag = "" if c.reachable else "   <- right arm cannot reach"
    print(f"   {name}  x={c.x:+.4f}  y={c.y:+.4f}  {c.shoulder_distance_m:.3f} m{flag}")

table surface z : 0.7400 m
rig rails       : 5

cells (row 1 = nearest the robot, col 1 = its LEFT / +y):
   r1c1  x=+0.2794  y=+0.1524  0.489 m
   r1c2  x=+0.2794  y=+0.0000  0.397 m
   r1c3  x=+0.2794  y=-0.1524  0.351 m
   r2c1  x=+0.4318  y=+0.1524  0.589 m
   r2c2  x=+0.4318  y=+0.0000  0.516 m
   r2c3  x=+0.4318  y=-0.1524  0.481 m
   r3c1  x=+0.5842  y=+0.1524  0.709 m   <- right arm cannot reach
   r3c2  x=+0.5842  y=+0.0000  0.649 m   <- right arm cannot reach
   r3c3  x=+0.5842  y=-0.1524  0.622 m


## 2. The pool

Ten objects, all on the floor at `y = ±0.75` or `±0.95` — far outside the right
arm's 0.609 m reach, so nothing can disturb them by accident.

They sit **on** the floor rather than hidden below it.  The floor is an infinite
plane, so a body parked at `z = -0.5` is penetrating it and gets ejected —
measured, one came back to `z = +0.005` carrying 2.4 m/s after 2000 steps.  On
the floor they rest quietly and they render, so the pool is visible rather than
magic.

In [3]:
from placement import shoulder_distance

print(f"{'object':16s} {'class':10s} {'shape':22s} where it waits")
for oid in pool_ids(doc):
    obj = next(o for o in doc["objects"] if o["id"] == oid)
    g = obj["geometry"]
    shape = (f"box {g['size']}" if g["kind"] == "box"
             else f"cyl r={g['radius']} l={g['length']}")
    x, y, z = obj["pose"]["position"]
    print(f"{oid:16s} {obj['semantic_class']:10s} {shape:22s} "
          f"({x:+.2f},{y:+.2f})  {shoulder_distance(x, y, z):.2f} m away")

on_board = [o["id"] for o in doc["objects"]
            if "manipulable" in (o.get("tags") or [])
            and any(abs(o["pose"]["position"][0] - c.x) <= c.half_extent
                    and abs(o["pose"]["position"][1] - c.y) <= c.half_extent
                    for c in cells.values())]
print(f"\nobjects on the board at startup: {on_board or 'none — the board is empty'}")

object           class      shape                  where it waits
red_cube         cube       box [0.06, 0.06, 0.06] (+0.15,+0.95)  1.47 m away
blue_cylinder    cylinder   cyl r=0.035 l=0.1      (+0.35,+0.95)  1.49 m away
soda_can         can        cyl r=0.033 l=0.115    (+0.55,+0.95)  1.55 m away
foam_block       foam       box [0.07, 0.07, 0.05] (+0.15,-0.95)  1.21 m away
pool_box_1       cube       box [0.04, 0.04, 0.04] (+0.15,+0.75)  1.33 m away
pool_box_2       cube       box [0.04, 0.04, 0.04] (+0.35,+0.75)  1.37 m away
pool_box_3       cube       box [0.04, 0.04, 0.04] (+0.55,+0.75)  1.43 m away
pool_cyl_1       cylinder   cyl r=0.02 l=0.08      (+0.15,-0.75)  1.08 m away
pool_cyl_2       cylinder   cyl r=0.02 l=0.08      (+0.35,-0.75)  1.12 m away
pool_cyl_3       cylinder   cyl r=0.02 l=0.08      (+0.55,-0.75)  1.20 m away

objects on the board at startup: none — the board is empty


## 3. A client

`place` waits for the ack rather than firing and hoping.  The ack carries
`sim_step` — the step the object actually landed on — so a later camera frame can
be lined up against a placement whose position you already know.  That is what
makes this a perception check rather than a demo.

Acks are routed back to the connection that asked.  Worth knowing because they
were not, briefly: a single shared queue meant the Docker bridge could swallow a
notebook's ack, and the notebook would wait forever for a placement that had
already happened.

In [4]:
class Pool:
    def __init__(self, ws):
        self.ws = ws

    async def _ack(self, rid):
        while True:
            msg = json.loads(await self.ws.recv())
            if msg["type"] == "place_ack" and msg["request_id"] == rid:
                return msg

    async def place(self, object_id, cell, yaw_deg=0.0, reshape=None, **kw):
        rid = f"{object_id}@{cell}"
        await self.ws.send(PlaceObject(object_id=object_id, cell=cell,
                                       yaw_deg=yaw_deg, reshape=reshape,
                                       request_id=rid, **kw).encode())
        ack = await self._ack(rid)
        if not ack["accepted"]:
            raise RuntimeError(ack["error"])
        return ack

    async def recall(self, object_id):
        """Back to the floor, out of Reachy's way."""
        rid = f"{object_id}@pool"
        await self.ws.send(PlaceObject(object_id=object_id, cell=None,
                                       request_id=rid).encode())
        return await self._ack(rid)

    async def poses(self):
        while True:
            msg = json.loads(await self.ws.recv())
            if msg["type"] == "state":
                return {o["object_id"]: o["pos_xyz"] for o in msg["objects"]}


async def connected(fn):
    async with websockets.connect(URI, max_size=None) as ws:
        await ws.send(Hello().encode()); await ws.recv()
        return await fn(Pool(ws))

## 4. Put something in front of Reachy

`r1c3` is the near-right cell — the one the right arm reaches most easily, at
0.351 m.

In [5]:
async def _one(pool):
    ack = await pool.place("red_cube", "r1c3")
    p = ack["placement"]
    print(f"red_cube -> {p['cell']}  pos={[round(v, 4) for v in p['position']]}  "
          f"step={ack['sim_step']}")

await connected(_one)

red_cube -> r1c3  pos=[0.2794, -0.1524, 0.772]  step=7591


## 5. Set up a scene for a task

Objects are refused when a cell is already taken, so a layout is built one cell
at a time.  Two bodies spawned into the same space start deeply interpenetrating
and MuJoCo resolves that by launching them — which would read as a physics bug
rather than as the caller's mistake.

In [6]:
LAYOUT = [("red_cube",      "r1c3",  0),
          ("soda_can",      "r2c2",  0),
          ("foam_block",    "r1c1", 20),
          ("blue_cylinder", "r2c3",  0)]

async def _setup(pool):
    for oid, _, _ in LAYOUT:
        await pool.recall(oid)
    for oid, cell, yaw in LAYOUT:
        ack = await pool.place(oid, cell, yaw_deg=yaw)
        print(f"  {oid:14s} -> {ack['placement']['cell']}  yaw={yaw}")

    poses = await pool.poses()
    print()
    for oid, cell, _ in LAYOUT:
        x, y, z = poses[oid]
        c = cells[cell]
        ok = abs(x - c.x) <= c.half_extent and abs(y - c.y) <= c.half_extent
        print(f"  {oid:14s} ({x:+.4f}, {y:+.4f}, {z:.4f})  on {cell}: {ok}")

await connected(_setup)

  red_cube       -> r1c3  yaw=0
  soda_can       -> r2c2  yaw=0
  foam_block     -> r1c1  yaw=20
  blue_cylinder  -> r2c3  yaw=0

  red_cube       (+0.2794, -0.1524, 0.7700)  on r1c3: True
  soda_can       (+0.4318, +0.0000, 0.7974)  on r2c2: True
  foam_block     (+0.2794, +0.1524, 0.7648)  on r1c1: True
  blue_cylinder  (+0.4318, -0.1524, 0.7896)  on r2c3: True


## 6. The spare slots, and reshaping them

The six `pool_*` slots are generic boxes and cylinders.  `geom_size`,
`geom_rgba` and `body_mass` are all writable on a compiled model, so one slot
becomes whatever a run needs without a recompile.

**Always go through `reshape`, never write `geom_size` yourself.**  `geom_rbound`
is the broadphase bounding radius, it is computed by the compiler, and it is
*not* recomputed when `geom_size` changes — `mj_setConst` does not fix it either.
Grow a geom without updating rbound and the broadphase culls the pair before
narrowphase runs, so contacts silently stop being generated.  That presents as
the gripper passing through the object, with nothing in any log.

In [7]:
async def _spares(pool):
    # a 4 cm grey cube — the size a real cube measures against a 12.7 cm cell
    await pool.place("pool_box_1", "r2c1",
                     reshape={"size": [0.04, 0.04, 0.04],
                              "rgba": [0.46, 0.46, 0.45, 1.0],
                              "mass": 0.05})
    # the same slot type, standing in for something taller and lighter
    await pool.place("pool_cyl_1", "r3c3",
                     reshape={"radius": 0.025, "length": 0.12, "mass": 0.02})
    print("two spare slots reshaped and placed")

    poses = await pool.poses()
    print()
    print("everything on the board now:")
    for oid, (x, y, z) in sorted(poses.items()):
        if z < 0.5:
            continue
        cell = next((n for n, c in cells.items()
                     if abs(x - c.x) <= c.half_extent
                     and abs(y - c.y) <= c.half_extent), "?")
        print(f"   {oid:16s} {cell}  ({x:+.4f}, {y:+.4f}, {z:.4f})")

await connected(_spares)

two spare slots reshaped and placed

everything on the board now:
   blue_cylinder    r2c3  (+0.4318, -0.1524, 0.7900)
   foam_block       r1c1  (+0.2794, +0.1524, 0.7650)
   pool_box_1       r2c1  (+0.4318, +0.1524, 0.7599)
   pool_cyl_1       r3c3  (+0.5842, -0.1524, 0.7994)
   red_cube         r1c3  (+0.2794, -0.1524, 0.7700)
   soda_can         r2c2  (+0.4318, +0.0000, 0.7975)


## 7. What it refuses

Each refusal comes back as an ack with a message, not as a dropped request — a
placement that silently does nothing is the failure mode this is built to avoid.

In [8]:
async def _refusals(pool):
    for oid, cell, why in [
        ("pool_box_2", "r3c1", "unreachable — 71 cm, past the arm's measured reach"),
        ("pool_box_2", "r1c3", "occupied — red_cube is already there"),
        ("no_such_thing", "r1c1", "unknown object"),
    ]:
        try:
            await pool.place(oid, cell)
            print(f"{oid:14s} {cell}  PLACED")
        except RuntimeError as exc:
            print(f"{oid:14s} {cell}  refused: {str(exc)[:74]}")
            print(f"{'':14s}      ({why})")

await connected(_refusals)

pool_box_2     r3c1  refused: cell r3c1 is 71 cm from the right shoulder and was measured UNREACHABLE by
                    (unreachable — 71 cm, past the arm's measured reach)
pool_box_2     r1c3  refused: cell r1c3 already holds 'red_cube'; stow it first, or pass allow_occupied=
                    (occupied — red_cube is already there)


no_such_thing  r1c1  refused: 'no_such_thing' is not a placeable object; the scene's free-joint objects 
                    (unknown object)


Both guards take an override, because both refusals are sometimes the point —
testing that a planner *declines* an unreachable target, or deliberately staging
a collision.

In [9]:
async def _override(pool):
    await pool.place("pool_box_2", "r3c1", allow_unreachable=True)
    print("placed on r3c1 deliberately — the arm still cannot reach it,")
    print("which is what makes it a useful test for a planner that should decline")
    await pool.recall("pool_box_2")
    print("recalled to the pool")

await connected(_override)

placed on r3c1 deliberately — the arm still cannot reach it,
which is what makes it a useful test for a planner that should decline
recalled to the pool


## 8. Clear the board

In [10]:
async def _clear(pool):
    for oid in pool_ids(doc):
        await pool.recall(oid)
    poses = await pool.poses()
    still_up = [o for o, (_, _, z) in poses.items() if z > 0.5]
    print("board clear" if not still_up else f"still on the board: {still_up}")

await connected(_clear)

board clear


## Ground still to cover

- **Placement is not a grasp.**  Reaching a cell and grasping at it are different
  constraints — MCC's sweep is orientation-unconstrained, and the tuned demo
  workspace was only y in [-0.22, +0.02].  Every col-1 cell is suspect for
  grasping even though the tip reaches it.
- **Object sizes are ~1.5x too large** (issue #42), which sets gripper aperture
  and approach clearance, not just how big things look.  The spare slots above
  are placed at 4 cm precisely because `reshape()` lets a run try the corrected
  number before anyone commits it to the scene.
- **`cell_r3c3` reachability is unresolved** (issue #41): 0.622 m, labelled
  reachable against a stated 0.609 m maximum.